# Découvrir les microservices OpenEO avec le client python
<br/>

Import et installation des librairies nécessaires

In [ ]:
try:
    import openeo
except:
    !pip install openeo
    import openeo

Connexion à l'aggrégateur :

In [ ]:
con = openeo.connect("https://aggregator.geodes-lab.fr")

## 1. Microservice Account Management
**Ce microservice s'occupe de l'authentification.**  
Une authentification HTTP Basic et OpenID-Connect est possible.<br/>

Dans le cas de l'aggrégateur CNES seule **l'authentification OIDC** est supportée :
- Lors de la première authentification, il faut cliquer sur le lien proposé qui va ouvrir une pop-up avec le portail d'authentification. Une fois les informations renseignées, le jeton est sauvegardé.
- Les autres authentifications sont automatiques et réutilisent le refresh token. 

In [ ]:
# Authentification OIDC dynamique
con.authenticate_oidc() 

D'autres méthodes de connexion sont disponibles. Pour plus de détails : *https://open-eo.github.io/openeo-python-client/auth.html*

In [ ]:
# Récupération des informations de l'utilisateur connecté
con.describe_account()

## 2. Microservice Capabilities
**Ce microservice permet d'obtenir des informations sur le backend** telles que :
- Les versions supportées et la version par défaut.
- Les microservices disponibles
- Les fichiers de formats supportés en entrée et sortie

In [ ]:
# Informations générales
con.capabilities()

In [ ]:
# Versions
con.version_info()

In [ ]:
# Formats supportés avec les options associées
con.list_file_formats()

In [ ]:
# URL de la plus haute version supportée
con.version_discovery(url="https://aggregator.geodes-lab.fr")

## 3. Microservice EO Data Discovery
**Ce microservice permet d'obtenir des informations sur les collections disponibles sur le backend.**  
Les collections sont des collections STAC.

Dans le cadre de cet atelier, une seule collection est disponible. Vous pouvez explorer davantage le contenu de la collection via le STAC Browser suivant : *https://browser.geodes-lab.fr/collections/S2-L2A-COG*

In [ ]:
# Lister les id de collections disponibles
con.list_collection_ids()

In [ ]:
# Pour avoir un peu plus d'informations sur les collections disponibles
con.list_collections()

In [ ]:
# Obtenir la description complète d'une collection
con.describe_collection("S2-L2A-COG")

## 4. Microservice Process Discovery
**Ce microservice permet d'obtenir des informations sur les processus disponibles sur le backend.**

In [ ]:
# Lister tous les processus disponibles
con.list_processes()

## 5. Microservice Data Processing
**Ce microservice permet de traiter les données.**

Plus explicitement, il permet de gérer les process graphs et jobs OpenEO créés par l'utilisateur. *Attention, les Jobs OpenEO qui ne sont pas les jobs SLURM exécutés sur TREX, même s'ils sont liés.*

**_Validation des process graphs_**

La validation d'un process graph est exécutée automatiquement lors de chaque création de job OpenEO. Cependant il est possible de la tester directement :

In [ ]:
# Validation d'un process graph valide
process_graph_test = {
    "process_graph": {
      "load": {
        "process_id": "load_collection",
        "description": "Loading the data",
        "arguments": {
          "id": "S2-L2A-COG",
          "spatial_extent": {
            "east": 1.52,
            "north": 43.67,
            "south": 43.53,
            "west": 1.35
          },
          "temporal_extent": ["2022-01-01", "2022-02-01"],
          "bands": ["B08", "B04", "B02"]
        }
      },
      "ndvi": {
        "process_id": "ndvi",
        "description": "NDVI calculation",
        "arguments": {
          "data": {"from_node": "load"} 
        },
      },
      "save": {
        "process_id": "save_result",
        "arguments": {
          "data": {"from_node": "ndvi"},
          "format": "GTiff"
        },
        "result": True
      }
    }
}

con.validate_process_graph(process_graph_test)

In [ ]:
process_graph2 = {
    "process_graph": {
      "load": {
        "process_id": "load_collection",
        "description": "Loading the data; The order of the specified bands is important for the following reduce operation.",
        "arguments": {
          "id": "S2-L2A-COG",
          "spatial_extent": {
            "east": 1.52,
            "north": 43.67,
            "south": 43.53,
            "west": 1.35
          },
          "temporal_extent": ["2022-01-01", "2022-02-01"],
          "bands": ["B08", "B04", "B02"]
        }
      },
      "mintime": {
        "process_id": "reduce_dimension",
        "description": "Compute a minimum time composite by reducing the temporal dimension",
        "arguments": {
          "data": {"from_node": "loadbl"},
          "dimension": "temporal",
          "reducer": {
            "process_graph": {
              "min": {
                "process_id": "ndvi",
                "arguments": {
                  "data": {"from_argument": "data"}
                },
                "result": True
              }
            }
          }
        }
      },
      "save": {
        "process_id": "save_result",
        "arguments": {
          "data": {"from_node": "mintime"},
          "format": "GTiff"
        },
        "result": True
      }
    }
}

con.validate_process_graph(process_graph2)

**_Gestion des jobs OpenEO_**

Tout l'enjeu d'OpenEO repose sur la facilitation de la récupération de données d'observation de la Terre puis de les traiter avec des traitements prédéfinis.  

Cela fonctionne avec la création de jobs dits OpenEO auxquels sont associés des traitements chaînés.  
Chaque traitement commence généralement par un *load_collection* (pour charger la donnée) et termine par un *save_result* (pour écrire le résultat). 

Le préfixe devant le nom du job renseigne à quel backend il est relié.

In [ ]:
# Lister ses jobs
con.list_jobs()

Exemple de création d'un job OpenEO :

In [ ]:
# Job très simple qui charge une image et l'écrit en format GTiff
df = con.load_collection(
    "S2-L2A-COG", 
    temporal_extent=["2022-01-10","2022-01-20"], 
    spatial_extent={"west": 1.33, "south": 43.53, "east": 1.54, "north": 43.67, "crs": 4326},  # bbox around Toulouse
    bands=["B2"]  # one band
)

# Création du job
job_df = df.create_job(title="Job exemple 1", out_format="Gtiff", compress="LZW")
job_df

Lors de la première exécution du job, une **estimation** est faite afin de déterminer les ressources dont le job SLURM va avoir besoin.  
Il est possible de déclencher uniquement l'estimation depuis le backend mais pas depuis l'aggrégateur.

Les jobs ne peuvent actuellement être exécutés qu'en mode asychrone. La méthode *start* permet de lancer l'exécution en fond alors que la fonction *start_and_wait* utilisée ici va attendre la fin d'exécution du job.

In [ ]:
# On lance l'exécution
job_df.start_and_wait()

In [ ]:
# On peut le retrouver dans la liste des jobs
con.list_jobs()

In [ ]:
# On peut lire les logs du jobs
job_df.logs()

# NOTE : ça prend parfois quelques secondes avant de s'afficher

In [ ]:
# On peut obtenir des infos sur les résultats
results = job_df.get_results()
results

In [ ]:
# Puis les télécharger
results.download_files("./results_job1/")

Les résultats comprennent toujours les fichiers de sortie générés ainsi qu'un fichier json qui récapitule les paramètres d'entrée du process graph exécuté.  

On peut maintenant supprimer le job OpenEO :

In [ ]:
# Supprimer le job
job_df.delete()

# Et vérifier sa suppression
con.list_jobs()

## 6. Microservice User Defined Processes
**Ce microservice gère les UDP (User Defined Processes).**

Une UDP est un process créé à partir d'autres process existants. Autrement dit, il s'agit d'un process graph préenregistré pouvant être réutilisé par l'utilisateur.

Dans cet example, nous allons créer une UDP permettant de convertir des degrés Fahrenheit en degrés Celsius.

In [ ]:
from openeo.api.process import Parameter
from openeo.processes import subtract, divide

# On définit les paramètres
f = Parameter.number("f", description="Degrees Fahrenheit.")

# On définit le process
fahrenheit_to_celsius = divide(x=subtract(x=f, y=32), y=1.8)

# On crée l'udp
con.save_user_defined_process(
    "fahrenheit_to_celsius",
    fahrenheit_to_celsius,
    parameters=[f]
)

In [ ]:
# Lister toutes ses UDP
con.list_user_defined_processes()

In [ ]:
# Update process (ajout description)

con.save_user_defined_process(
    "fahrenheit_to_celsius",
    fahrenheit_to_celsius,
    parameters=[f],
    summary="Fahrenheit to celsius"
)

In [ ]:
# Suppression de l'UDP
udp_fahrenheit = con.user_defined_process("fahrenheit_to_celsius")
udp_fahrenheit.delete()

# Vérification de la suppression de l'UDP
con.list_user_defined_processes()